In [ ]:
import requests
import numpy as np
import pandas as pd
from datetime import datetime
import os
import time

import os
from dotenv import load_dotenv
load_dotenv()

base_url = os.getenv("BASE_URL")


latitude_center, longitude_center = 46.1512, 14.9955
lat_min = latitude_center - 2
lat_max = latitude_center + 2
lon_min = longitude_center - 2
lon_max = longitude_center + 2

lat_values = np.linspace(lat_min, lat_max, int((lat_max - lat_min) / 0.018))
lon_values = np.linspace(lon_min, lon_max, int((lon_max - lon_min) / 0.018))

data = []

start_date = "2023-08-16"
end_date = "2023-08-19"

print("Starting to fetch data...")
for i, lat in enumerate(lat_values):
    for j, lon in enumerate(lon_values):
        url = f"https://{base_url}/v1/dwd-icon?latitude={lat}&longitude={lon}&daily=precipitation_sum&timezone=Europe%2FLondon&start_date={start_date}&end_date={end_date}"
        response = requests.get(url)
        daily_precipitation = response.json()['daily']['precipitation_sum']

        # Sum all the precipitation values
        total_precipitation = sum(daily_precipitation)

        data.append([lat, lon, total_precipitation])

        progress_percentage = (i * len(lon_values) + j + 1) / (len(lat_values) * len(lon_values)) * 100
        print(f"Progress: {progress_percentage:.2f}% complete")

print("Data fetching complete.")

df = pd.DataFrame(data, columns=["Latitude", "Longitude", "Total_Precipitation"])
timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
filename = f'precipitation_data_{start_date}_to_{end_date}_{timestamp}.csv'
df.to_csv(filename, index=False)
print(f"Data saved to '{filename}'")
